In [ ]:
import pandas as pd
import numpy as np

# Dicionário de Pesos de Posição (Exemplo)
POS_WEIGHTS = {
    'GK': -1, 'LB': -1, 'LCB': -1, 'CB': -1, 'RCB': -1, 'RB': -1, 'LWB': -1, 'RWB': -1,
    'DM': 0, 'LDM': 0, 'RDM': 0, 'CM': 0, 'LCM': 0, 'RCM': 0, 'LM': 0, 'RM': 0, 'AM': 0,
    'LW': 1, 'RW': 1, 'CF': 1, 'ST': 1, 'SS': 1
}

def classificar_janela_substituicao(df_subs_janela):
    """
    Classifica se a janela foi Ofensiva, Defensiva ou Neutra.
    (Requer que seu dataframe tenha colunas indicando a posição, ex: 'pos_off', 'pos_on'.
     Se não tiver, você pode adaptar para buscar no players.json)
    """
    saldo_tatico = 0
    for _, sub in df_subs_janela.iterrows():
        # Fallback para 'CM' (Neutro) caso a posição não seja encontrada
        pos_out = sub.get('player_off_pos', 'CM') 
        pos_in = sub.get('player_on_pos', 'CM')
        
        peso_out = POS_WEIGHTS.get(pos_out, 0)
        peso_in = POS_WEIGHTS.get(pos_in, 0)
        saldo_tatico += (peso_in - peso_out)
        
    if saldo_tatico > 0: return 'Ofensiva'
    elif saldo_tatico < 0: return 'Defensiva'
    else: return 'Neutra'

def avaliar_janela_modelos(t_start, t_end, team_id, df_actions, subs_players_on):
    """
    Calcula o Plus-Minus para Gols (Baseline), xG, xT e VAEP.
    Retorna um dicionário com os scores normalizados por minuto.
    """
    duracao_min = (t_end - t_start) / 60.0
    if duracao_min <= 0:
        return { 'PM_Gols': 0, 'PM_xG': 0, 'PM_xT': 0, 'PM_VAEP': 0, 'PM_Ponderado': 0, 'Duracao_Min': 0 }
        
    # Filtra as ações da janela
    acoes = df_actions[(df_actions['start_time'] >= t_start) & (df_actions['start_time'] < t_end)]
    
    acoes_time = acoes[acoes['team_id'] == team_id]
    acoes_adv = acoes[acoes['team_id'] != team_id]

    # Cálculos dos Deltas Brutos (Time - Adversário)
    delta_gols = acoes_time['goal'].sum() - acoes_adv['goal'].sum()
    delta_xg = acoes_time.get('xG', pd.Series(0, index=acoes_time.index)).sum() - acoes_adv.get('xG', pd.Series(0, index=acoes_adv.index)).sum()
    delta_xt = acoes_time['xt_added'].sum() - acoes_adv['xt_added'].sum()
    delta_vaep = acoes_time['vaep_value'].sum() - acoes_adv['vaep_value'].sum()

    # Normalização pelo tempo (Taxa por minuto)
    score_gols = delta_gols / duracao_min
    score_xg = delta_xg / duracao_min
    score_xt = delta_xt / duracao_min
    score_vaep = delta_vaep / duracao_min

    # Regra de Final de Jogo (Aplica apenas às métricas avançadas, gols são sagrados)
    min_sub = t_start / 60.0
    if min_sub > 82:
        # Pega as ações exclusivas de quem entrou
        acoes_reservas = acoes_time[acoes_time['player_id'].isin(subs_players_on)]
        impacto_reservas = acoes_reservas['vaep_value'].sum()
        
        # Se os reservas mal tocaram na bola / não geraram valor, anula os scores avançados
        if abs(impacto_reservas) < 0.05:
            score_xg, score_xt, score_vaep = 0.0, 0.0, 0.0

    # Modelo Ponderado (Você pode ajustar esses pesos depois no seu estudo)
    # Ex: 50% VAEP (Geral), 30% xT (Ameaça/Posse), 20% xG (Finalização)
    score_ponderado = (score_vaep * 0.5) + (score_xt * 0.3) + (score_xg * 0.2)

    return {
        'Duracao_Min': duracao_min,
        'PM_Gols_Baseline': score_gols,
        'PM_xG': score_xg,
        'PM_xT': score_xt,
        'PM_VAEP': score_vaep,
        'PM_Ponderado': score_ponderado
    }

import pandas as pd

def avaliar_janela_modelos_ponderada(t_start, t_next_sub, t_end_game, team_id, df_actions, subs_players_on, peso_primario=1.0, peso_secundario=0.4):
    """
    Calcula o Plus-Minus utilizando duas janelas com pesos diferentes.
    - Primária: impacto direto até a próxima troca.
    - Secundária: impacto diluído até o fim do jogo.
    """
    # 1. Calcular as durações (em minutos)
    dur_primaria = (t_next_sub - t_start) / 60.0
    dur_secundaria = (t_end_game - t_next_sub) / 60.0
    
    duracao_ponderada = (dur_primaria * peso_primario) + (dur_secundaria * peso_secundario)
    
    if duracao_ponderada <= 0:
        return { 'PM_Gols_Baseline': 0, 'PM_xG': 0, 'PM_xT': 0, 'PM_VAEP': 0, 'PM_Ponderado': 0, 'Duracao_Real_Min': 0 }

    # 2. Separar as ações nos dois blocos de tempo
    acoes_p = df_actions[(df_actions['start_time'] >= t_start) & (df_actions['start_time'] < t_next_sub)]
    acoes_s = df_actions[(df_actions['start_time'] >= t_next_sub) & (df_actions['start_time'] <= t_end_game)]

    # Função interna auxiliar para extrair o saldo (Team - Adv) de um bloco de ações
    def calcular_saldos(df_bloco):
        if df_bloco.empty:
            return {'gols': 0, 'xg': 0, 'xt': 0, 'vaep': 0}
            
        time = df_bloco[df_bloco['team_id'] == team_id]
        adv = df_bloco[df_bloco['team_id'] != team_id]
        
        return {
            'gols': time['goal'].sum() - adv['goal'].sum(),
            'xg': time.get('xG', pd.Series(0, index=time.index)).sum() - adv.get('xG', pd.Series(0, index=adv.index)).sum(),
            'xt': time['xt_added'].sum() - adv['xt_added'].sum(),
            'vaep': time['vaep_value'].sum() - adv['vaep_value'].sum()
        }

    # 3. Pegar os saldos das duas janelas
    saldo_p = calcular_saldos(acoes_p)
    saldo_s = calcular_saldos(acoes_s)

    # 4. Calcular a Taxa Ponderada por Minuto para cada métrica
    score_gols = ((saldo_p['gols'] * peso_primario) + (saldo_s['gols'] * peso_secundario)) / duracao_ponderada
    score_xg = ((saldo_p['xg'] * peso_primario) + (saldo_s['xg'] * peso_secundario)) / duracao_ponderada
    score_xt = ((saldo_p['xt'] * peso_primario) + (saldo_s['xt'] * peso_secundario)) / duracao_ponderada
    score_vaep = ((saldo_p['vaep'] * peso_primario) + (saldo_s['vaep'] * peso_secundario)) / duracao_ponderada

    # 5. Regra de Final de Jogo (Aplica-se ao impacto direto dos jogadores em todo o tempo restante)
    min_sub = t_start / 60.0
    if min_sub > 82:
        # Pega todas as ações dali até o fim do jogo
        acoes_restantes = df_actions[(df_actions['start_time'] >= t_start) & (df_actions['start_time'] <= t_end_game)]
        acoes_reservas = acoes_restantes[(acoes_restantes['team_id'] == team_id) & (acoes_restantes['player_id'].isin(subs_players_on))]
        
        impacto_reservas = acoes_reservas['vaep_value'].sum()
        
        # Se entraram e não fizeram nada (não geraram VAEP), zera os scores avançados
        if abs(impacto_reservas) < 0.05:
            score_xg, score_xt, score_vaep = 0.0, 0.0, 0.0

    # 6. Score Final Ponderado do Modelo
    score_ponderado = (score_vaep * 0.5) + (score_xt * 0.3) + (score_xg * 0.2)

    return {
        'Duracao_Real_Min': dur_primaria + dur_secundaria, # O tempo real até o fim do jogo
        'PM_Gols_Baseline': score_gols,
        'PM_xG': score_xg,
        'PM_xT': score_xt,
        'PM_VAEP': score_vaep,
        'PM_Ponderado': score_ponderado
    }

In [ ]:
import pandas as pd

def avaliar_janela_contextual(t_start, t_next_sub, t_end_game, team_id, df_actions, subs_players_on):
    """
    Avalia a substituição separando Valores Ofensivos e Defensivos e
    penalizando a Exposição ao Risco (xG e xT Adversário).
    """
    # 1. CÁLCULO DOS TEMPOS (Com Decaimento)
    dur_primaria = (t_next_sub - t_start) / 60.0
    dur_secundaria = (t_end_game - t_next_sub) / 60.0
    duracao_ponderada = (dur_primaria * 1.0) + (dur_secundaria * 0.4)
    
    if duracao_ponderada <= 0:
        return {'PM_Gols_Baseline': 0, 'PM_xG': 0, 'PM_xT': 0, 'PM_VAEP': 0, 'PM_Ponderado': 0, 'Duracao_Real_Min': 0, 'Game_State': 'Desconhecido'}

    # 2. LEITURA DO GAME STATE (PLACAR ATUAL)
    acoes_passadas = df_actions[df_actions['start_time'] < t_start]
    gols_pro = acoes_passadas[acoes_passadas['team_id'] == team_id]['goal'].sum()
    gols_contra = acoes_passadas[acoes_passadas['team_id'] != team_id]['goal'].sum()
    saldo_atual = gols_pro - gols_contra
    
    if saldo_atual > 0: status_jogo = 'Vencendo'
    elif saldo_atual < 0: status_jogo = 'Perdendo'
    else: status_jogo = 'Empatando'

    # 3. EXTRAÇÃO DAS MÉTRICAS SEPARADAS (Pró e Contra)
    acoes_p = df_actions[(df_actions['start_time'] >= t_start) & (df_actions['start_time'] < t_next_sub)]
    acoes_s = df_actions[(df_actions['start_time'] >= t_next_sub) & (df_actions['start_time'] <= t_end_game)]

    def extrair_metricas(df_bloco):
        if df_bloco.empty: 
            return {'gols_pro':0, 'gols_con':0, 'xg_pro':0, 'xg_con':0, 'xt_pro':0, 'xt_con':0, 'ovaep_pro':0, 'ovaep_con':0, 'dvaep_pro':0, 'dvaep_con':0}
            
        time = df_bloco[df_bloco['team_id'] == team_id]
        adv = df_bloco[df_bloco['team_id'] != team_id]
        
        return {
            'gols_pro': time['goal'].sum(), 
            'gols_con': adv['goal'].sum(),
            'xg_pro': time.get('xG', pd.Series(0, index=time.index)).sum(), 
            'xg_con': adv.get('xG', pd.Series(0, index=adv.index)).sum(),
            'xt_pro': time['xt_added'].sum(), 
            'xt_con': adv['xt_added'].sum(),
            'ovaep_pro': time['offensive_value'].sum(), 
            'ovaep_con': adv['offensive_value'].sum(),
            'dvaep_pro': time['defensive_value'].sum(), 
            'dvaep_con': adv['defensive_value'].sum()
        }

    met_p = extrair_metricas(acoes_p)
    met_s = extrair_metricas(acoes_s)

    # Função auxiliar para calcular a Taxa por Minuto de qualquer métrica com os pesos de tempo
    def calc_taxa(campo):
        return ((met_p[campo] * 1.0) + (met_s[campo] * 0.4)) / duracao_ponderada

    rate = {k: calc_taxa(k) for k in met_p.keys()}

    # Saldos Tradicionais (Apenas para exportação ao DataFrame)
    score_gols = rate['gols_pro'] - rate['gols_con']
    score_xg = rate['xg_pro'] - rate['xg_con']
    score_xt = rate['xt_pro'] - rate['xt_con']
    score_vaep = (rate['ovaep_pro'] + rate['dvaep_pro']) - (rate['ovaep_con'] + rate['dvaep_con'])

    # ==========================================
    # 4. PESOS DINÂMICOS & PENALIZAÇÃO DE EXPOSIÇÃO
    # ==========================================
    if status_jogo == 'Vencendo':
        # VENCENDO: A defesa é ouro. O ataque importa pouco.
        # Prêmio gigante para VAEP Defensivo próprio. Punição severa se o adversário conseguir gerar xG ou xT (Exposição).
        score_ponderado = (rate['dvaep_pro'] * 1.2) - (rate['xg_con'] * 0.6) - (rate['xt_con'] * 0.3) + (score_vaep * 0.2)
        
    elif status_jogo == 'Perdendo':
        # PERDENDO: É preciso atacar de forma incisiva (Volume e Chance).
        # Porém, se o time se desorganizar e tomar contra-ataques letais (xg_con), a nota é destruída.
        score_ponderado = (rate['ovaep_pro'] * 0.5) + (rate['xg_pro'] * 0.5) + (rate['xt_pro'] * 0.3) - (rate['xg_con'] * 0.8)
        
    else:
        # EMPATANDO: Modelo de domínio total. Subtraímos explicitamente as ameaças do adversário.
        saldo_ovaep = rate['ovaep_pro'] - rate['ovaep_con']
        saldo_dvaep = rate['dvaep_pro'] - rate['dvaep_con']
        score_ponderado = (saldo_ovaep * 0.4) + (saldo_dvaep * 0.4) + (score_xt * 0.2)

    # ==========================================
    # 5. REGRA DE GARBAGE TIME (Minutos Finais)
    # ==========================================
    min_sub = t_start / 60.0
    if min_sub >= 84: 
        acoes_restantes = df_actions[(df_actions['start_time'] >= t_start) & (df_actions['start_time'] <= t_end_game)]
        acoes_reservas = acoes_restantes[(acoes_restantes['team_id'] == team_id) & (acoes_restantes['player_id'].isin(subs_players_on))]
        
        fez_gol = acoes_reservas['goal'].sum() > 0
        impacto_vaep = acoes_reservas['vaep_value'].sum()
        
        # Se os reservas não ajudaram diretamente de forma mensurável, neutraliza a janela.
        if not fez_gol and abs(impacto_vaep) < 0.15:
            score_ponderado = 0.0
            score_vaep = 0.0
            score_xt = 0.0
            score_xg = 0.0

    return {
        'Game_State': status_jogo,
        'Duracao_Real_Min': dur_primaria + dur_secundaria,
        'PM_Gols_Baseline': score_gols,
        'PM_xG': score_xg,
        'PM_xT': score_xt,
        'PM_VAEP': score_vaep,
        'PM_Ponderado': score_ponderado
    }

In [ ]:
def gerar_dataset_modelos_substituicao(actions_df, subs_df):
    """
    Motor que varre os jogos gerando as notas considerando a janela dupla.
    """
    resultados = []
    jogos = subs_df['match_id'].unique()
    
    print(f"Processando {len(jogos)} jogos para o modelo com Decaimento Temporal...")
    
    for game_id in jogos:
        df_game_actions = actions_df[actions_df['match_id'] == game_id].copy()
        df_game_subs = subs_df[subs_df['match_id'] == game_id].copy()
        
        if df_game_actions.empty or df_game_subs.empty:
            continue
            
        t_end_game = df_game_actions['start_time'].max()
        
        # Agrupar substituições simultâneas (< 30 segundos de diferença)
        df_game_subs = df_game_subs.sort_values(by=['team_id', 'start_time'])
        df_game_subs['time_diff'] = df_game_subs.groupby('team_id')['start_time'].diff().fillna(0)
        df_game_subs['window_id'] = (df_game_subs['time_diff'] > 30).astype(int).groupby(df_game_subs['team_id']).cumsum()
        
        janelas = df_game_subs.groupby(['team_id', 'window_id'])
        
        for (team_id, w_id), sub_grupo in janelas:
            t_start = sub_grupo['start_time'].min()
            jogadores_in = sub_grupo['player_on_id'].tolist()
            
            # Localiza qual é a próxima substituição (t_next_sub)
            subs_futuras = df_game_subs[(df_game_subs['team_id'] == team_id) & (df_game_subs['start_time'] > t_start + 30)]
            t_next_sub = subs_futuras['start_time'].min() if not subs_futuras.empty else t_end_game
            
            perfil_tatico = classificar_janela_substituicao(sub_grupo)
            
            notas = avaliar_janela_contextual(
                t_start=t_start, 
                t_next_sub=t_next_sub, 
                t_end_game=t_end_game, 
                team_id=team_id, 
                df_actions=df_game_actions, 
                subs_players_on=jogadores_in
            )
            
            if notas['Duracao_Real_Min'] >= 1.0:
                resultados.append({
                    'match_id': game_id,
                    'team_id': team_id,
                    'start_time': t_start,
                    'minuto_jogo': t_start / 60.0,
                    'jogadores_entraram': len(jogadores_in),
                    'perfil_tatico': perfil_tatico,
                    'Game_State': notas['Game_State'], # <- ADICIONADO AQUI
                    'duracao_total_min': notas['Duracao_Real_Min'],
                    'PM_Gols_Baseline': notas['PM_Gols_Baseline'],
                    'PM_xG': notas['PM_xG'],
                    'PM_xT': notas['PM_xT'],
                    'PM_VAEP': notas['PM_VAEP'],
                    'PM_Ponderado': notas['PM_Ponderado']
                })

    df_resultados = pd.DataFrame(resultados)
    print("✅ Dataset de Modelos Ponderados gerado com sucesso!")
    return df_resultados

In [ ]:
# Gera o dataframe com as notas de todos os modelos
df_modelos_subs = gerar_dataset_modelos_substituicao(actions_final_df, substitutions_df)

# Exibe uma amostra dos resultados
display(df_modelos_subs.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def analisar_correlacao_modelos(df):
    print("🔍 ANÁLISE 1: MATRIZ DE CORRELAÇÃO DAS MÉTRICAS")
    
    # Seleciona apenas as colunas numéricas de score
    colunas_score = ['PM_Gols_Baseline', 'PM_xG', 'PM_xT', 'PM_VAEP', 'PM_Ponderado']
    df_corr = df[colunas_score].corr()
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, 
                cbar_kws={'label': 'Nível de Correlação'})
    plt.title("O GOL vs O DESEMPENHO REAL\nCorrelação entre as métricas de Substituição", pad=20)
    plt.tight_layout()
    plt.show()

# Execute:
analisar_correlacao_modelos(df_modelos_subs)

In [ ]:
import plotly.express as px

def analisar_perfil_tatico(df):
    print("📊 ANÁLISE 2: IMPACTO POR PERFIL TÁTICO")
    
    # Agrupa por perfil e calcula a média das métricas avançadas
    df_perfil = df.groupby('perfil_tatico')[['PM_VAEP', 'PM_xT', 'PM_Ponderado']].mean().reset_index()
    
    # Formata a tabela para ficar em formato longo (melhor para o Plotly)
    df_melt = df_perfil.melt(id_vars='perfil_tatico', var_name='Metrica', value_name='Score_Medio')
    
    fig = px.bar(
        df_melt, 
        x='perfil_tatico', 
        y='Score_Medio', 
        color='Metrica', 
        barmode='group',
        title='Qual o impacto real de "Colocar o time pra frente"?',
        labels={'Score_Medio': 'Score Médio Ponderado por Minuto', 'perfil_tatico': 'Intenção do Treinador'},
        color_discrete_map={'PM_VAEP': '#1f77b4', 'PM_xT': '#ff7f0e', 'PM_Ponderado': '#2ca02c'}
    )
    
    # Adiciona uma linha no ZERO para separar quem piora e quem melhora o time
    fig.add_hline(y=0, line_width=2, line_dash="dash", line_color="black")
    fig.update_layout(template='plotly_white')
    fig.show(renderer='browser')

# Execute:
analisar_perfil_tatico(df_modelos_subs)

In [ ]:
import pandas as pd

def analisar_momento_do_jogo(df):
    print("⏱️ ANÁLISE 3: A HORA CERTA DE MEXER")
    
    # Cria faixas de tempo do jogo
    bins = [0, 45, 60, 75, 90, 120]
    labels = ['1º Tempo', 'Volta do Intervalo (45-60)', 'Decisão (60-75)', 'Reta Final (75-90)', 'Acréscimos']
    
    df_copy = df.copy()
    df_copy['fase_jogo'] = pd.cut(df_copy['minuto_jogo'], bins=bins, labels=labels, right=False)
    
    # Avalia quantas substituições deram CERTO (Score Ponderado Positivo) vs ERRADO
    df_copy['Deu_Certo'] = (df_copy['PM_Ponderado'] > 0).astype(int)
    
    resumo_fase = df_copy.groupby('fase_jogo').agg(
        Total_Subs=('match_id', 'count'),
        Score_Medio=('PM_Ponderado', 'mean'),
        Taxa_Sucesso_Pct=('Deu_Certo', lambda x: (x.mean() * 100).round(1))
    ).reset_index()
    
    display(resumo_fase)

# Execute:
analisar_momento_do_jogo(df_modelos_subs)

In [ ]:
def ranking_impacto_banco(df, min_janelas=5):
    print("🏆 ANÁLISE 4: O POWER RANKING DOS TIMES/TREINADORES")
    
    # Filtra times com amostragem pequena
    contagem = df['team_id'].value_counts()
    times_validos = contagem[contagem >= min_janelas].index
    df_validos = df[df['team_id'].isin(times_validos)]
    
    ranking = df_validos.groupby('team_id').agg(
        Total_Janelas_Avaliadas=('duracao_total_min', 'count'),
        Impacto_VAEP=('PM_VAEP', 'mean'),
        Impacto_xT=('PM_xT', 'mean'),
        Nota_Final_Treinador=('PM_Ponderado', 'mean')
    ).reset_index()
    
    # Ordena pelos melhores do campeonato
    ranking = ranking.sort_values('Nota_Final_Treinador', ascending=False).reset_index(drop=True)
    
    # Arredonda para ficar limpo
    cols_arredondar = ['Impacto_VAEP', 'Impacto_xT', 'Nota_Final_Treinador']
    ranking[cols_arredondar] = ranking[cols_arredondar].round(4)
    
    # Adiciona rank
    ranking.index = ranking.index + 1
    
    display(ranking.head(15)) # Mostra os 15 melhores
    return ranking

# Execute:
df_ranking = ranking_impacto_banco(df_modelos_subs)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score

def avaliar_qualidade_modelo_avancado(df_modelos):
    print("=" * 80)
    print("🏆 AVALIAÇÃO MULTIDIMENSIONAL DO MODELO DE SUBSTITUIÇÕES")
    print("=" * 80)
    
    # Filtro de segurança: Janelas muito curtas (< 5 min) têm muito ruído
    df_eval = df_modelos[df_modelos['duracao_total_min'] >= 5.0].copy()
    
    # =========================================================
    # 1. MATRIZ DE CORRELAÇÃO (O que a nota final realmente reflete?)
    # =========================================================
    print("\n📈 1. Correlações com a Nota do Treinador (PM_Ponderado)")
    cor_gols = df_eval['PM_Ponderado'].corr(df_eval['PM_Gols_Baseline'])
    cor_xg = df_eval['PM_Ponderado'].corr(df_eval['PM_xG'])
    cor_xt = df_eval['PM_Ponderado'].corr(df_eval['PM_xT'])
    
    print(f" -> Correlação com Gols Reais:   {cor_gols:.3f} (Sempre será baixo devido ao acaso)")
    print(f" -> Correlação com Saldo de xG:  {cor_xg:.3f} (O ideal é ser moderado/alto)")
    print(f" -> Correlação com Domínio (xT): {cor_xt:.3f}")

    # =========================================================
    # 2. TAXA DE ACERTO (Direção do Jogo em xG)
    # =========================================================
    print("\n🎯 2. Taxa de Acerto de Domínio (Acurácia baseada em xG)")
    # Realidade justa: O time gerou mais chances do que sofreu? (PM_xG > 0)
    y_real_xg = (df_eval['PM_xG'] > 0).astype(int)
    
    # O modelo disse que a janela foi positiva?
    y_pred = (df_eval['PM_Ponderado'] > 0.01).astype(int)
    
    acuracia_xg = accuracy_score(y_real_xg, y_pred)
    precisao_xg = precision_score(y_real_xg, y_pred, zero_division=0)
    
    print(f" -> Acurácia Global (Acertou quem criaria mais chances): {acuracia_xg*100:.1f}%")
    print(f" -> Precisão (Se a nota foi BOA, o time dominou as chances?): {precisao_xg*100:.1f}%")

    # =========================================================
    # 3. TESTE DE EXTREMOS MULTIMÉTRICAS (O Gráfico da Verdade)
    # =========================================================
    print("\n⚖️ 3. Avaliação dos Extremos (Quintis de Nota)")
    
    # Divide as notas do modelo em 5 prateleiras
    df_eval['Quintil_Modelo'] = pd.qcut(df_eval['PM_Ponderado'], 5, labels=['1-Péssimo', '2-Ruim', '3-Neutro', '4-Bom', '5-Excelente'])
    
    # Avalia as consequências de cada prateleira
    extremos = df_eval.groupby('Quintil_Modelo', observed=True).agg(
        Media_Gols=('PM_Gols_Baseline', 'mean'),
        Media_xG=('PM_xG', 'mean'),
        Media_xT=('PM_xT', 'mean')
    ).reset_index()
    
    display(extremos)
    
    # Plotagem dupla (Gols vs xG)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Gráfico 1: A realidade dura (Gols)
    sns.barplot(data=extremos, x='Quintil_Modelo', y='Media_Gols', palette='RdYlGn', ax=axes[0])
    axes[0].set_title('Consequência em GOLS REAIS')
    axes[0].axhline(0, color='black', linewidth=1)
    axes[0].set_ylabel('Saldo de Gols / Min')
    
    # Gráfico 2: A realidade tática (xG)
    sns.barplot(data=extremos, x='Quintil_Modelo', y='Media_xG', palette='RdYlGn', ax=axes[1])
    axes[1].set_title('Consequência em CHANCES CRIADAS (xG)')
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_ylabel('Saldo de xG / Min')
    
    plt.suptitle("Validação do Modelo: O que acontece quando o modelo dá uma nota?", fontsize=16)
    plt.tight_layout()
    plt.show()

# Execute passando o seu dataframe:
avaliar_qualidade_modelo_avancado(df_modelos_subs)

In [ ]:
import pandas as pd
import numpy as np

def create_substitution_windows(subs_df, actions_df):
    """
    Agrupa substituições que ocorrem quase ao mesmo tempo no mesmo time
    e define os tempos de início (t_start) e fim (t_end) de cada janela.
    """
    # Ordena as substituições por jogo, time e tempo
    df_subs = subs_df.sort_values(by=['match_id', 'team_id', 'start_time']).copy()
    
    # Agrupa substituições do mesmo time que ocorrem com menos de 30 segundos de diferença
    df_subs['time_diff'] = df_subs.groupby(['match_id', 'team_id'])['start_time'].diff().fillna(0)
    df_subs['window_id'] = (df_subs['time_diff'] > 30).astype(int).groupby([df_subs['match_id'], df_subs['team_id']]).cumsum()
    
    # Pega o tempo final de cada jogo
    match_end_times = actions_df.groupby('match_id')['start_time'].max().to_dict()
    
    windows = []
    # Agrupa por janela
    grouped = df_subs.groupby(['match_id', 'team_id', 'window_id'])
    
    for (match_id, team_id, w_id), group in grouped:
        t_start = group['start_time'].min()
        players_in = group['player_on_id'].tolist()
        players_out = group['player_off_id'].tolist()
        
        # Encontra a próxima janela do MESMO time neste jogo
        next_subs = df_subs[(df_subs['match_id'] == match_id) & 
                            (df_subs['team_id'] == team_id) & 
                            (df_subs['start_time'] > t_start + 30)]
                            
        t_end = next_subs['start_time'].min() if not next_subs.empty else match_end_times.get(match_id, 5400)
        
        windows.append({
            'match_id': match_id,
            'team_id': team_id,
            'window_id': w_id,
            't_start': t_start,
            't_end': t_end,
            'num_subs': len(players_in),
            'players_in': players_in,
            'players_out': players_out
        })
        
    return pd.DataFrame(windows)

def calculate_plus_minus_models(windows_df, actions_df):
    """
    Calcula o Plus-Minus bruto e o Score Categórico (1, 0, -1) para Gols, xG, xT e VAEP.
    """
    # Garante que a coluna xG exista nas ações, preenchendo com 0 se não houver
    if 'xG' not in actions_df.columns:
        actions_df['xG'] = 0.0

    results = []
    
    for _, window in windows_df.iterrows():
        match_id = window['match_id']
        team_id = window['team_id']
        t_start = window['t_start']
        t_end = window['t_end']
        
        # Filtra as ações dentro dessa janela de tempo e partida
        window_actions = actions_df[
            (actions_df['match_id'] == match_id) & 
            (actions_df['start_time'] >= t_start) & 
            (actions_df['start_time'] < t_end)
        ]
        
        if window_actions.empty:
            continue
            
        # Separa ações do Time vs Ações do Adversário
        team_actions = window_actions[window_actions['team_id'] == team_id]
        opp_actions = window_actions[window_actions['team_id'] != team_id]
        
        # 1. VALORES BRUTOS (PLUS-MINUS)
        pm_goals = team_actions['goal'].sum() - opp_actions['goal'].sum()
        pm_xg = team_actions['xG'].sum() - opp_actions['xG'].sum()
        pm_xt = team_actions['xt_added'].sum() - opp_actions['xt_added'].sum()
        pm_vaep = team_actions['vaep_value'].sum() - opp_actions['vaep_value'].sum()
        
        # Função auxiliar para classificar: >0 -> 1 (Boa), <0 -> -1 (Ruim), ==0 -> 0 (Neutra)
        # Para métricas contínuas (xG, xT, VAEP), usamos uma pequena margem (epsilon) 
        # para evitar classificar "0.0001" como uma mudança drástica.
        def classify_score(val, epsilon=0.05):
            if val > epsilon: return 1
            elif val < -epsilon: return -1
            else: return 0
            
        results.append({
            'match_id': match_id,
            'team_id': team_id,
            'window_id': window['window_id'],
            'minuto_sub': round(t_start / 60, 1),
            'duracao_janela_min': round((t_end - t_start) / 60, 1),
            'num_subs': window['num_subs'],
            
            # Plus-Minus Bruto
            'pm_goals_raw': pm_goals,
            'pm_xg_raw': pm_xg,
            'pm_xt_raw': pm_xt,
            'pm_vaep_raw': pm_vaep,
            
            # Modelo Classificador Binário/Neutro (1, 0, -1)
            'score_goals': classify_score(pm_goals, epsilon=0), # Gols não precisa de epsilon
            'score_xg': classify_score(pm_xg, epsilon=0.01),
            'score_xt': classify_score(pm_xt, epsilon=0.05),
            'score_vaep': classify_score(pm_vaep, epsilon=0.15)
        })
        
    return pd.DataFrame(results)

# Exemplo de uso (assumindo que 'actions_final_df' já tem a coluna 'xG' que você mapeou no código anterior):
df_windows = create_substitution_windows(substitutions_df, actions_final_df)
df_pm_models = calculate_plus_minus_models(df_windows, actions_final_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def evaluate_baseline_models(pm_df):
    """
    Avalia a aderência dos modelos de xG, xT e VAEP em relação ao baseline de Gols.
    """
    print("="*60)
    print("📊 AVALIAÇÃO DOS MODELOS BASELINE DE SUBSTITUIÇÃO")
    print("="*60)
    
    # Filtra janelas muito curtas (ex: < 3 minutos) onde o acaso domina
    valid_windows = pm_df[pm_df['duracao_janela_min'] >= 3.0].copy()
    print(f"Total de Janelas de Substituição válidas (>= 3 min): {len(valid_windows)}\n")
    
    # Dicionário de mapeamento para facilitar a leitura
    label_map = {1: 'Boa (+)', 0: 'Neutra (=)', -1: 'Ruim (-)'}
    
    # Distribuição das Classificações para cada Modelo
    modelos = ['score_goals', 'score_xg', 'score_xt', 'score_vaep']
    
    print("1️⃣ DISTRIBUIÇÃO DAS CLASSIFICAÇÕES POR MODELO:")
    dist_df = pd.DataFrame()
    for mod in modelos:
        dist_df[mod] = valid_windows[mod].map(label_map).value_counts(normalize=True) * 100
    display(dist_df.fillna(0).round(1).astype(str) + '%')
    print("\n*Nota: Gols tendem a ter muitas janelas 'Neutras' (0x0). Métricas de performance tendem a ser mais decisivas.\n")

    # Matriz de Concordância com o Baseline (Gols)
    print("2️⃣ TAXA DE CONCORDÂNCIA COM O BASELINE (GOLS):")
    # Ignorando os casos onde Gols = 0, para ver se quando sai gol, o modelo concorda com o lado
    decisive_goals = valid_windows[valid_windows['score_goals'] != 0]
    
    if not decisive_goals.empty:
        for mod in ['score_xg', 'score_xt', 'score_vaep']:
            # Taxa de Acerto Direcional (O modelo X percebeu quem dominou e fez o gol?)
            acertos = (decisive_goals['score_goals'] == decisive_goals[mod]).mean() * 100
            print(f"Acerto Direcional do modelo {mod.replace('score_', '').upper()}: {acertos:.1f}%")
    else:
        print("Amostra pequena demais de janelas com gols para medir concordância direta.")
        
    # Correlações dos Valores Brutos (Raw)
    print("\n3️⃣ CORRELAÇÃO DE PEARSON (Valores Contínuos):")
    raw_cols = ['pm_goals_raw', 'pm_xg_raw', 'pm_xt_raw', 'pm_vaep_raw']
    corr_matrix = valid_windows[raw_cols].corr().round(3)
    display(corr_matrix)

    # Plotando Matriz de Confusão: VAEP vs Baseline de Gols
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(valid_windows['score_goals'], valid_windows['score_vaep'], labels=[1, 0, -1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Boa (VAEP)', 'Neutra (VAEP)', 'Ruim (VAEP)'],
                yticklabels=['Boa (Gols)', 'Neutra (Gols)', 'Ruim (Gols)'])
    plt.title("Matriz de Confusão: Modelo Gols vs Modelo VAEP")
    plt.xlabel("Classificação do Modelo VAEP")
    plt.ylabel("Classificação Baseline (Gols)")
    plt.show()

# Chamada da função (após gerar df_pm_models)
evaluate_baseline_models(df_pm_models)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import seaborn as sns

def gerar_modelos_avancados(df_pm_models, df_lineups=None):
    """
    df_pm_models: O seu dataframe atual.
    df_lineups: Um dataframe hipotético ligando 'match_id' e 'minuto_sub' aos IDs dos 22 jogadores em campo.
    """
    print("="*60)
    print("🚀 GERANDO MODELOS AVANÇADOS: xG P90 e RAPM")
    print("="*60)

    # ==========================================
    # MODELO 1: xG Plus-Minus (Taxa por 90 min)
    # ==========================================
    # Evita divisão por zero em janelas inválidas
    df_valid = df_pm_models[df_pm_models['duracao_janela_min'] > 0].copy()
    
    # Criando métricas P90 (Per 90 minutes)
    for metric in ['goals', 'xg', 'xt', 'vaep']:
        raw_col = f'pm_{metric}_raw'
        p90_col = f'pm_{metric}_p90'
        # Fórmula: (Valor Bruto / Minutos da Janela) * 90
        df_valid[p90_col] = (df_valid[raw_col] / df_valid['duracao_janela_min']) * 90

    print("✅ Modelo 1 Concluído: Métricas normalizadas para P90.")
    display(df_valid[['duracao_janela_min', 'pm_xg_raw', 'pm_xg_p90']].head())


    # ==========================================
    # MODELO 2: RAPM (Regularized Adjusted Plus-Minus)
    # ==========================================
    # NOTA: Para rodar de verdade, você precisa da Matriz X (Jogadores).
    # Como não temos os lineups aqui, vou criar uma função robusta que você chamará 
    # após montar sua matriz de jogadores.
    
    def executar_rapm(X, y, pesos_minutos, nomes_jogadores, alpha=1000):
        """
        X: Matriz esparsa (Janelas x Jogadores) contendo +1 (time), -1 (adversário), 0 (fora).
        y: A métrica alvo (ex: df_valid['pm_xg_p90']).
        pesos_minutos: A duração da janela (df_valid['duracao_janela_min']).
        nomes_jogadores: Lista com o nome ou ID correspondente a cada coluna da matriz X.
        alpha: Força da penalização L2 (Ridge). Ajuste via Cross-Validation.
        """
        print(f"\nTreinando modelo RAPM com penalidade Ridge (Alpha={alpha})...")
        
        # O fit_intercept=False é padrão em RAPM pois a baseline teórica de um jogo 
        # entre times iguais é uma diferença de 0.
        rapm_model = Ridge(alpha=alpha, fit_intercept=False, solver='lsqr')
        
        # Treinando o modelo ponderado pelo tempo da janela (janelas longas importam mais)
        rapm_model.fit(X, y, sample_weight=pesos_minutos)
        
        # Extraindo o impacto (coeficientes) de cada jogador
        df_resultados_rapm = pd.DataFrame({
            'Jogador': nomes_jogadores,
            'RAPM_Rating': rapm_model.coef_
        })
        
        # Ordenando do maior impacto positivo ao menor
        df_resultados_rapm = df_resultados_rapm.sort_values(by='RAPM_Rating', ascending=False).reset_index(drop=True)
        
        return df_resultados_rapm, rapm_model

    print("\n⚠️ Para executar o Modelo 2 (RAPM), prepare sua Matriz 'X' cruzando as janelas com os jogadores em campo e utilize a função 'executar_rapm(X, y, pesos)'.")
    
    return df_valid, executar_rapm

# Execução no seu ambiente:
df_pm_avancado, func_rapm = gerar_modelos_avancados(df_pm_models)

In [ ]:
def classify_score(val, epsilon):
    if val > epsilon: return 1
    elif val < -epsilon: return -1
    else: return 0

def gerar_scores_p90(df_valid):
    """
    Gera scores categóricos baseados nas métricas P90 usando limiares adaptados.
    """
    # Gols não precisa de P90 para o score, o saldo bruto de gols já dita se a janela foi boa ou ruim.
    # Mas se quiser usar, o epsilon continua sendo 0.
    df_valid['score_goals_p90'] = df_valid['pm_goals_p90'].apply(lambda x: classify_score(x, epsilon=0))
    
    # Epsilon de xG P90: Um time precisa gerar um ritmo de pelo menos 0.25 xG de vantagem por jogo
    # para considerarmos que ele realmente dominou a janela.
    df_valid['score_xg_p90'] = df_valid['pm_xg_p90'].apply(lambda x: classify_score(x, epsilon=0.25))
    
    # Epsilon de xT P90: xT acumula mais rápido que xG. Um diferencial de 0.50 P90 é um bom filtro.
    df_valid['score_xt_p90'] = df_valid['pm_xt_p90'].apply(lambda x: classify_score(x, epsilon=0.50))
    
    # Epsilon de VAEP P90: VAEP tem valores mais altos por envolver ações defensivas também.
    df_valid['score_vaep_p90'] = df_valid['pm_vaep_p90'].apply(lambda x: classify_score(x, epsilon=1.0))
    
    return df_valid

# Aplicando ao seu dataframe
df_pm_avancado = gerar_scores_p90(df_pm_avancado)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from IPython.display import display # Adicionado para garantir o display no Jupyter/Colab

def evaluate_advanced_models(pm_df):
    """
    Avalia a aderência dos modelos de xG, xT e VAEP em P90 em relação ao baseline de Gols.
    """
    print("="*60)
    print("📊 AVALIAÇÃO DOS MODELOS AVANÇADOS (P90)")
    print("="*60)
    
    # Filtra janelas muito curtas (ex: < 3 minutos) onde o acaso domina
    valid_windows = pm_df[pm_df['duracao_janela_min'] >= 3.0].copy()
    print(f"Total de Janelas de Substituição válidas (>= 3 min): {len(valid_windows)}\n")
    
    # Dicionário de mapeamento para facilitar a leitura
    label_map = {1: 'Boa (+)', 0: 'Neutra (=)', -1: 'Ruim (-)'}
    
    # Distribuição das Classificações para cada Modelo
    modelos = ['score_goals_p90', 'score_xg_p90', 'score_xt_p90', 'score_vaep_p90']
    
    print("1️⃣ DISTRIBUIÇÃO DAS CLASSIFICAÇÕES POR MODELO:")
    dist_df = pd.DataFrame()
    for mod in modelos:
        dist_df[mod] = valid_windows[mod].map(label_map).value_counts(normalize=True) * 100
    display(dist_df.fillna(0).round(1).astype(str) + '%')
    print("\n*Nota: Gols tendem a ter muitas janelas 'Neutras' (0x0). Métricas de performance tendem a ser mais decisivas.\n")

    # Matriz de Concordância com o Baseline (Gols)
    print("2️⃣ TAXA DE CONCORDÂNCIA COM O BASELINE (GOLS):")
    # Ignorando os casos onde Gols = 0, para ver se quando sai gol, o modelo concorda com o lado
    decisive_goals = valid_windows[valid_windows['score_goals_p90'] != 0]
    
    if not decisive_goals.empty:
        for mod in ['score_xg_p90', 'score_xt_p90', 'score_vaep_p90']:
            # Taxa de Acerto Direcional (O modelo X percebeu quem dominou e fez o gol?)
            acertos = (decisive_goals['score_goals'] == decisive_goals[mod]).mean() * 100
            print(f"Acerto Direcional do modelo {mod.replace('score_', '').upper()}: {acertos:.1f}%")
    else:
        print("Amostra pequena demais de janelas com gols para medir concordância direta.")
        
    # Correlações dos Valores Contínuos P90 (MUDANÇA PRINCIPAL AQUI)
    print("\n3️⃣ CORRELAÇÃO DE PEARSON (Valores Contínuos P90):")
    p90_cols = ['pm_goals_p90', 'pm_xg_p90', 'pm_xt_p90', 'pm_vaep_p90']
    
    # Calcula e exibe a matriz de correlação usando as novas colunas
    corr_matrix = valid_windows[p90_cols].corr().round(3)
    display(corr_matrix)

    # Plotando Matriz de Confusão: VAEP vs Baseline de Gols
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(valid_windows['score_goals_p90'], valid_windows['score_vaep_p90'], labels=[1, 0, -1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Boa (VAEP)', 'Neutra (VAEP)', 'Ruim (VAEP)'],
                yticklabels=['Boa (Gols)', 'Neutra (Gols)', 'Ruim (Gols)'])
    plt.title("Matriz de Confusão: Modelo Gols vs Modelo VAEP (Avançado)")
    plt.xlabel("Classificação do Modelo VAEP")
    plt.ylabel("Classificação Baseline (Gols)")
    plt.show()

In [ ]:
# Chamada da função para o seu dataframe atualizado
evaluate_advanced_models(df_pm_avancado)

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import lil_matrix

print("="*60)
print("🛠️ CONSTRUINDO A MATRIZ DE DESIGN RAPM")
print("="*60)

# 1. Obter todos os jogadores únicos que realizaram alguma ação
todos_jogadores = actions_df['player_id'].dropna().unique()
num_jogadores = len(todos_jogadores)

# Criar um dicionário para mapear player_id para o índice da coluna na matriz
player_to_idx = {player: idx for idx, player in enumerate(todos_jogadores)}
idx_to_player = {idx: player for player, idx in player_to_idx.items()}

print(f"Total de jogadores únicos no dataset: {num_jogadores}")

In [ ]:
def mapear_jogadores_por_janela(df_windows, actions_df):
    """
    Deduz os titulares e mapeia os 22 jogadores em campo para cada janela.
    """
    janelas_com_jogadores = []
    
    # Agrupar por partida
    partidas = df_windows['match_id'].unique()
    
    for match in partidas:
        # Pega as janelas e ações desta partida
        windows_match = df_windows[df_windows['match_id'] == match].sort_values('t_start')
        actions_match = actions_df[actions_df['match_id'] == match]
        
        times = actions_match['team_id'].dropna().unique()
        if len(times) != 2:
            continue # Pula jogos com erro de dados
            
        time_A, time_B = times[0], times[1]
        
        # --- DEDUÇÃO DOS TITULARES ---
        # Todos os jogadores que atuaram pelo time na partida
        jogaram_A = set(actions_match[actions_match['team_id'] == time_A]['player_id'].dropna())
        jogaram_B = set(actions_match[actions_match['team_id'] == time_B]['player_id'].dropna())
        
        # Todos os jogadores que ENTRARAM como substitutos na partida
        subs_in_A = set([p for sub_list in windows_match[windows_match['team_id'] == time_A]['players_in'] for p in sub_list])
        subs_in_B = set([p for sub_list in windows_match[windows_match['team_id'] == time_B]['players_in'] for p in sub_list])
        
        # Titulares = Jogaram - Entraram como reserva
        em_campo_A = jogaram_A - subs_in_A
        em_campo_B = jogaram_B - subs_in_B
        
        # --- ITERAÇÃO NAS JANELAS ---
        for _, window in windows_match.iterrows():
            time_janela = window['team_id']
            
            # Atualiza o estado ANTES de gravar a janela (processa a substituição que abriu a janela)
            # Nota: Na sua lógica original, a janela começa no exato momento da substituição.
            if time_janela == time_A:
                for p_out in window['players_out']: em_campo_A.discard(p_out)
                for p_in in window['players_in']: em_campo_A.add(p_in)
            else:
                for p_out in window['players_out']: em_campo_B.discard(p_out)
                for p_in in window['players_in']: em_campo_B.add(p_in)
            
            # Salva o estado atual da janela
            # Precisamos saber quem é o "Time Foco" (+1) e quem é o "Adversário" (-1)
            if time_janela == time_A:
                jogadores_time = list(em_campo_A)
                jogadores_adv = list(em_campo_B)
            else:
                jogadores_time = list(em_campo_B)
                jogadores_adv = list(em_campo_A)
                
            janelas_com_jogadores.append({
                'match_id': match,
                'team_id': time_janela,
                'window_id': window['window_id'],
                'jogadores_time': jogadores_time,
                'jogadores_adv': jogadores_adv
            })
            
    return pd.DataFrame(janelas_com_jogadores)

# Executa o mapeamento
df_lineups = mapear_jogadores_por_janela(df_windows, actions_df)
print("✅ Mapeamento de jogadores em campo concluído.")

In [ ]:
# Faz o merge das escalações com as suas métricas avançadas já calculadas
df_rapm_ready = pd.merge(df_pm_avancado, df_lineups, on=['match_id', 'team_id', 'window_id'])

# Filtra apenas janelas válidas (ex: > 0 minutos, ou > 3 minutos se preferir)
df_rapm_ready = df_rapm_ready[df_rapm_ready['duracao_janela_min'] > 0].reset_index(drop=True)
num_janelas = len(df_rapm_ready)

# Inicializa a matriz esparsa (Janelas x Jogadores)
X = lil_matrix((num_janelas, num_jogadores))

print("Preenchendo a Matriz X (+1 para time, -1 para adversário)...")

for i, row in df_rapm_ready.iterrows():
    # +1 para os jogadores do time da casa/foco
    for p in row['jogadores_time']:
        if p in player_to_idx:
            X[i, player_to_idx[p]] = 1
            
    # -1 para os jogadores do time adversário
    for p in row['jogadores_adv']:
        if p in player_to_idx:
            X[i, player_to_idx[p]] = -1

# Converter para formato CSR (mais eficiente para matemática/machine learning)
X = X.tocsr()

print(f"✅ Matriz X pronta! Dimensões: {X.shape[0]} janelas x {X.shape[1]} jogadores.")

In [ ]:
from sklearn.linear_model import Ridge

# 1. Definir a variável alvo (y) e os pesos
y_xg = df_rapm_ready['pm_xg_p90'].values
pesos = df_rapm_ready['duracao_janela_min'].values

# 2. Executar o modelo
print("\nTreinando modelo RAPM para Expected Goals (xG)...")
rapm_model = Ridge(alpha=1000, fit_intercept=False, solver='lsqr')
rapm_model.fit(X, y_xg, sample_weight=pesos)

# 3. Organizar os resultados
nomes_jogadores = [idx_to_player[i] for i in range(num_jogadores)]

df_resultados_rapm = pd.DataFrame({
    'player_id': nomes_jogadores,
    'RAPM_xG': rapm_model.coef_
})

# Ordenar do melhor para o pior
df_resultados_rapm = df_resultados_rapm.sort_values(by='RAPM_xG', ascending=False).reset_index(drop=True)

print("\n🏆 TOP 5 Jogadores (Impacto xG P90):")
display(df_resultados_rapm.head())

print("\n🔻 BOTTOM 5 Jogadores (Impacto xG P90):")
display(df_resultados_rapm.tail())

In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

# 1. Carregar o arquivo JSON
CAMINHO_PLAYERS = Path('dados/') / 'players.json'
with open(CAMINHO_PLAYERS, 'r', encoding='utf-8') as f:
    players_list = json.load(f)

# 2. Converter para DataFrame
df_players = pd.DataFrame(players_list)

# 3. Isolar as colunas
coluna_id_json = 'id'
coluna_nome_json = 'nickname'

df_players_reduzido = df_players[[coluna_id_json, coluna_nome_json]].copy()

df_players_reduzido = df_players_reduzido.rename(columns={
    coluna_id_json: 'player_id',
    coluna_nome_json: 'player_name'
})

# ==========================================
# 4. Harmonizar os tipos de dados e LIMPAR AMBOS OS LADOS
# ==========================================
# Forçamos a conversão para numérico primeiro
df_players_reduzido['player_id'] = pd.to_numeric(df_players_reduzido['player_id'], errors='coerce')
df_resultados_rapm['player_id'] = pd.to_numeric(df_resultados_rapm['player_id'], errors='coerce')

# Removemos eventuais NaNs
df_players_reduzido = df_players_reduzido.dropna(subset=['player_id'])
df_resultados_rapm = df_resultados_rapm.dropna(subset=['player_id'])

# Convertemos para o tipo Inteiro do Pandas
df_players_reduzido['player_id'] = df_players_reduzido['player_id'].astype('Int64')
df_resultados_rapm['player_id'] = df_resultados_rapm['player_id'].astype('Int64')

# ------------------------------------------
# 🛠️ REMOÇÃO DE DUPLICATAS EM AMBOS OS DFS
# ------------------------------------------
# Garante que o dicionário de nomes tenha apenas 1 linha por ID
df_players_reduzido = df_players_reduzido.drop_duplicates(subset=['player_id'], keep='first')

# CRUCIAL: Garante que o resultado do RAPM também tenha apenas 1 linha por ID (remove resíduos da memória)
df_resultados_rapm = df_resultados_rapm.drop_duplicates(subset=['player_id'], keep='first')


# ==========================================
# 5. Fazer o cruzamento (Merge) - COM LIMPEZA PRÉVIA
# ==========================================
# Se a coluna player_name já existir no df do RAPM, nós a removemos antes do merge
if 'player_name' in df_resultados_rapm.columns:
    df_resultados_rapm = df_resultados_rapm.drop(columns=['player_name'])

# Agora o merge acontece de forma 1-para-1 estrita
df_resultados_rapm_limpo = pd.merge(
    df_resultados_rapm, 
    df_players_reduzido, 
    on='player_id', 
    how='left' 
)

df_resultados_rapm_limpo = df_resultados_rapm_limpo.reset_index(drop=True)

# ==========================================
# 6. Exibir o resultado final limpo
# ==========================================
if 'player_name' in df_resultados_rapm_limpo.columns:
    df_resultados_rapm_limpo = df_resultados_rapm_limpo[['player_id', 'player_name', 'RAPM_xG']]
    
    print("🏆 TOP 10 Jogadores (Impacto xG P90):")
    display(df_resultados_rapm_limpo.head(10))

    print("\n🔻 BOTTOM 10 Jogadores (Impacto xG P90):")
    display(df_resultados_rapm_limpo.tail(10))
else:
    print("❌ ERRO INESPERADO. Colunas atuais:", df_resultados_rapm_limpo.columns.tolist())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("="*60)
print("⚖️ COMPARANDO BASELINE (PLUS-MINUS) vs RAPM")
print("="*60)

# ==========================================
# 1. Expandir o Baseline para o nível do Jogador
# ==========================================
# Vamos ler o df_rapm_ready e atribuir os saldos para cada jogador em campo
linhas_jogadores = []

for _, row in df_rapm_ready.iterrows():
    # Saldo positivo para os jogadores do time principal
    for p in row['jogadores_time']:
        linhas_jogadores.append({
            'player_id': p, 
            'minutos': row['duracao_janela_min'], 
            'pm_xg_p90': row['pm_xg_p90']
        })
    # Saldo invertido (negativo) para os jogadores do time adversário
    for p in row['jogadores_adv']:
        linhas_jogadores.append({
            'player_id': p, 
            'minutos': row['duracao_janela_min'], 
            'pm_xg_p90': -row['pm_xg_p90']
        })

df_jogadores_pm = pd.DataFrame(linhas_jogadores)

# ==========================================
# 2. Calcular a Média Ponderada (Baseline P90)
# ==========================================
# Calculamos a média do Plus-Minus pesando pelos minutos de cada janela
def calcular_media_ponderada(x):
    total_min = x['minutos'].sum()
    if total_min > 0:
        pm_medio = (x['pm_xg_p90'] * x['minutos']).sum() / total_min
    else:
        pm_medio = 0
    return pd.Series({'minutos_totais': total_min, 'baseline_pm_xg': pm_medio})

df_baseline = df_jogadores_pm.groupby('player_id').apply(calcular_media_ponderada).reset_index()

# Forçando o tipo para Int64 para um merge seguro
df_baseline['player_id'] = pd.to_numeric(df_baseline['player_id'], errors='coerce').astype('Int64')

# ==========================================
# 3. Juntar Baseline e RAPM
# ==========================================
df_comparacao = pd.merge(df_resultados_rapm_limpo, df_baseline, on='player_id', how='inner')

# Filtrar apenas jogadores com minutos relevantes (ex: mais de 90 minutos)
# Isso limpa a visualização e remove quem jogou apenas 5 minutos no campeonato inteiro
df_plot = df_comparacao[df_comparacao['minutos_totais'] >= 90].copy()

# ==========================================
# 4. Avaliação Estatística e Visual
# ==========================================
# Correlação
correlacao = df_plot['baseline_pm_xg'].corr(df_plot['RAPM_xG'])
print(f"📊 Correlação entre Baseline e RAPM: {correlacao:.3f}")
print("Nota: Uma correlação alta (~0.7 a 0.9) é esperada, mas as divergências são o mais importante!\n")

# Gráfico de Dispersão (Scatter Plot)
plt.figure(figsize=(10, 8))

# Linha de tendência/identidade 
sns.regplot(
    data=df_plot, 
    x='baseline_pm_xg', 
    y='RAPM_xG', 
    scatter=False, 
    color='gray', 
    line_kws={'linestyle':'--', 'alpha':0.5}
)

# Plotando os jogadores
sns.scatterplot(
    data=df_plot, 
    x='baseline_pm_xg', 
    y='RAPM_xG', 
    size='minutos_totais',
    sizes=(20, 300),
    alpha=0.7,
    edgecolor='k',
    hue='RAPM_xG', # Colore baseado na nota do RAPM
    palette='coolwarm',
    legend=False
)

# Destacar alguns jogadores específicos (Top 3 e Bottom 3 do RAPM)
top_bottom = pd.concat([df_plot.nlargest(3, 'RAPM_xG'), df_plot.nsmallest(3, 'RAPM_xG')])
for _, row in top_bottom.iterrows():
    plt.text(row['baseline_pm_xg'] + 0.01, row['RAPM_xG'], row['player_name'], fontsize=9)

# Formatação visual
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=1)
plt.title("Comparação: Plus-Minus Tradicional vs RAPM (Efeito Ridge)", fontsize=14, fontweight='bold')
plt.xlabel("Baseline Plus-Minus (xG P90 Médio)", fontsize=12)
plt.ylabel("RAPM_xG (Score Pós-Regressão)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import scipy.stats as stats

print("="*60)
print("🚀 EXPANDINDO RAPM E PADRONIZANDO (Z-SCORE)")
print("="*60)

# 1. Extrair os novos Targets (y) do seu df_rapm_ready
y_xt = df_rapm_ready['pm_xt_p90'].values
y_vaep = df_rapm_ready['pm_vaep_p90'].values
pesos = df_rapm_ready['duracao_janela_min'].values

# 2. Treinar os novos modelos Ridge (usando a mesma matriz X que você já construiu)
print("Treinando modelo RAPM para xT...")
rapm_xt_model = Ridge(alpha=1000, fit_intercept=False, solver='lsqr')
rapm_xt_model.fit(X, y_xt, sample_weight=pesos)

print("Treinando modelo RAPM para VAEP...")
rapm_vaep_model = Ridge(alpha=1000, fit_intercept=False, solver='lsqr')
rapm_vaep_model.fit(X, y_vaep, sample_weight=pesos)

# 3. Adicionar ao DataFrame de resultados limpo
df_resultados_rapm_limpo['RAPM_xT'] = rapm_xt_model.coef_
df_resultados_rapm_limpo['RAPM_VAEP'] = rapm_vaep_model.coef_

# Supondo que você também juntou o Baseline_PM_xG, Baseline_PM_xT, etc.
# Vamos aplicar o Z-Score em todas as métricas numéricas relevantes
metricas_para_padronizar = [
    'RAPM_xG', 'RAPM_xT', 'RAPM_VAEP', 
    'baseline_pm_xg' # Adicione os outros baselines aqui se tiver calculado
]

for col in metricas_para_padronizar:
    if col in df_resultados_rapm_limpo.columns:
        # Fórmula do Z-Score: (Valor - Média) / Desvio Padrão
        col_zscore = f'Z_{col}'
        df_resultados_rapm_limpo[col_zscore] = stats.zscore(df_resultados_rapm_limpo[col])

print("\n✅ Modelos treinados e dados padronizados!")
display(df_resultados_rapm_limpo[['player_name', 'Z_RAPM_xG', 'Z_RAPM_xT', 'Z_RAPM_VAEP']].head(10))

In [ ]:
from scipy.sparse import lil_matrix

def construir_matriz_rapm(df_janelas, num_jogadores_total, dict_player_to_idx):
    """Constrói a matriz X esparsa para um subconjunto de dados."""
    X_temp = lil_matrix((len(df_janelas), num_jogadores_total))
    
    for i, row in df_janelas.reset_index(drop=True).iterrows():
        for p in row['jogadores_time']:
            if p in dict_player_to_idx: X_temp[i, dict_player_to_idx[p]] = 1
        for p in row['jogadores_adv']:
            if p in dict_player_to_idx: X_temp[i, dict_player_to_idx[p]] = -1
            
    return X_temp.tocsr()

print("\n" + "="*60)
print("🧪 SPLIT-HALF TESTING: PODER PREDITIVO")
print("="*60)

# 1. Dividir o campeonato ao meio (assumindo que match_id seja sequencial ou você pode ordenar por data)
jogos_unicos = df_rapm_ready['match_id'].unique()
meio = len(jogos_unicos) // 2
jogos_h1 = jogos_unicos[:meio]
jogos_h2 = jogos_unicos[meio:]

# Separar DataFrames
df_h1 = df_rapm_ready[df_rapm_ready['match_id'].isin(jogos_h1)].copy()
df_h2 = df_rapm_ready[df_rapm_ready['match_id'].isin(jogos_h2)].copy()

# 2. Treinar o RAPM APENAS na Metade 1
X_h1 = construir_matriz_rapm(df_h1, num_jogadores, player_to_idx)
y_h1_xg = df_h1['pm_xg_p90'].values
pesos_h1 = df_h1['duracao_janela_min'].values

rapm_h1 = Ridge(alpha=500, fit_intercept=False, solver='lsqr') # Alpha um pouco menor pois a amostra caiu pela metade
rapm_h1.fit(X_h1, y_h1_xg, sample_weight=pesos_h1)

df_h1_resultados = pd.DataFrame({'player_id': list(idx_to_player.values()), 'RAPM_H1_xG': rapm_h1.coef_})

# 3. Calcular o Baseline P90 na Metade 1 (Como fizemos antes)
# (Resumo do cálculo de baseline para H1 - você pode usar a mesma lógica do código anterior)
# Vamos focar na essência: precisamos do Baseline PM da Metade 1.
# Assuma que geramos um dataframe `df_h1_baseline` com 'player_id' e 'Baseline_H1_xG'.

# 4. Calcular O FATO REAL na Metade 2 (O que realmente aconteceu)
# O target do H2 será o Saldo de Gols P90 REAL que o jogador presenciou quando em campo.
linhas_h2 = []
for _, row in df_h2.iterrows():
    for p in row['jogadores_time']: linhas_h2.append({'player_id': p, 'min': row['duracao_janela_min'], 'pm_goals_p90': row['pm_goals_p90']})
    for p in row['jogadores_adv']: linhas_h2.append({'player_id': p, 'min': row['duracao_janela_min'], 'pm_goals_p90': -row['pm_goals_p90']})

df_jogadores_h2 = pd.DataFrame(linhas_h2)
resultado_real_h2 = df_jogadores_h2.groupby('player_id').apply(
    lambda x: (x['pm_goals_p90'] * x['min']).sum() / x['min'].sum() if x['min'].sum() > 0 else 0
).rename('Real_Gols_H2').reset_index()

# 5. Juntar H1 (Predição) com H2 (Realidade)
# df_teste = pd.merge(df_h1_resultados, df_h1_baseline, on='player_id') 
df_teste = pd.merge(df_h1_resultados, resultado_real_h2, on='player_id', how='inner')

# Filtrar jogadores com minutagem mínima nas duas metades para não poluir o teste
# df_teste = df_teste[df_teste['minutos_h1'] > 180 & df_teste['minutos_h2'] > 180]

# 6. O VEREDITO: Qual métrica do passado prevê melhor o futuro?
corr_rapm = df_teste['RAPM_H1_xG'].corr(df_teste['Real_Gols_H2'])
# corr_baseline = df_teste['Baseline_H1_xG'].corr(df_teste['Real_Gols_H2'])

print(f"Desempenho Preditivo (Correlação com o Saldo de Gols Futuro):")
# print(f"Baseline Tradicional (Metade 1): {corr_baseline:.3f}")
print(f"RAPM Regularizado (Metade 1): {corr_rapm:.3f}")
print("\n*Nota: O modelo com a maior correlação ganha o 'Teste de Ouro' por ser mais confiável para scouts e diretores.*")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

print("="*60)
print("🧬 GERANDO ÍNDICE COMPOSTO (PCA): SCORE 0-100")
print("="*60)

# 1. Isolar as colunas que vão formar o índice (Os resultados do RAPM padronizados)
colunas_index = ['Z_RAPM_xG', 'Z_RAPM_xT', 'Z_RAPM_VAEP']

# Garantir que não há valores nulos que possam quebrar o PCA
df_pca_ready = df_resultados_rapm_limpo.dropna(subset=colunas_index).copy()
X_pca = df_pca_ready[colunas_index]

# 2. Executar o PCA (Queremos apenas o Componente Principal 1 - a "essência" da performance)
pca = PCA(n_components=1)
pca_scores = pca.fit_transform(X_pca)

# 3. Correção de Direção (O truque de Mestre)
# O PCA é cego; ele não sabe o que é "bom" ou "ruim", apenas encontra a maior variância.
# Às vezes, ele inverte o eixo matemático (valores mais negativos ficam sendo os melhores).
# Vamos checar a correlação do PCA com o VAEP. Se for negativa, nós invertemos o sinal do PCA.
if np.corrcoef(pca_scores.flatten(), X_pca['Z_RAPM_VAEP'])[0, 1] < 0:
    pca_scores = pca_scores * -1

df_pca_ready['PCA_Raw'] = pca_scores

# 4. Transformar em um Score Escalonado (0 a 100) para uso "Humano"
scaler = MinMaxScaler(feature_range=(0, 100))
df_pca_ready['Sub_Impact_Score'] = scaler.fit_transform(df_pca_ready[['PCA_Raw']])

# Arredondar para ficar bonito no relatório
df_pca_ready['Sub_Impact_Score'] = df_pca_ready['Sub_Impact_Score'].round(1)

# Ordenar os jogadores do Melhor para o Pior baseado no novo Score
df_final_ranking = df_pca_ready.sort_values(by='Sub_Impact_Score', ascending=False).reset_index(drop=True)

# 5. Entendendo o Modelo (Extraindo os Pesos)
# O seu treinador vai perguntar: "Do que é feito esse score?". Aqui está a resposta.
pesos_pca = pca.components_[0]
if np.corrcoef(pca_scores.flatten(), X_pca['Z_RAPM_VAEP'])[0, 1] < 0:
    pesos_pca = pesos_pca * -1 # Ajusta os pesos se o eixo foi invertido

print("\n⚖️ PESOS DO ÍNDICE (Composição da Nota):")
for metrica, peso in zip(colunas_index, pesos_pca):
    # Transforma em porcentagem aproximada de importância
    importancia = (abs(peso) / sum(abs(pesos_pca))) * 100
    print(f"🔹 {metrica.replace('Z_RAPM_', '')}: {importancia:.1f}%")

# 6. Exibir o Ranking Final
colunas_exibicao = ['player_name', 'Sub_Impact_Score', 'Z_RAPM_xG', 'Z_RAPM_xT', 'Z_RAPM_VAEP']
print("\n🏆 TOP 10 JOGADORES - IMPACTO DE SUBSTITUIÇÃO:")
display(df_final_ranking[colunas_exibicao].head(10))

print("\n🔻 BOTTOM 5 JOGADORES:")
display(df_final_ranking[colunas_exibicao].tail(5))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import scipy.stats as stats

def gerar_ranking_pca(df_rapm_base, df_janelas, players_json_list, min_minutos=5):
    """
    Filtra goleiros e minutagem baixa, recalcula Z-Scores e gera o Score PCA.
    """
    print("="*60)
    print(f"🧬 GERANDO ÍNDICE PCA (Filtro: >={min_minutos} min | Sem Goleiros)")
    print("="*60)
    
    # ==========================================
    # 1. Obter Minutos Totais por Jogador
    # ==========================================
    linhas_minutos = []
    for _, row in df_janelas.iterrows():
        for p in row['jogadores_time']: linhas_minutos.append({'player_id': p, 'minutos': row['duracao_janela_min']})
        for p in row['jogadores_adv']: linhas_minutos.append({'player_id': p, 'minutos': row['duracao_janela_min']})
        
    df_minutos = pd.DataFrame(linhas_minutos).groupby('player_id')['minutos'].sum().reset_index()
    df_minutos['player_id'] = pd.to_numeric(df_minutos['player_id'], errors='coerce').astype('Int64')

    # ==========================================
    # 2. Identificar e Remover Goleiros
    # ==========================================
    df_metadata = pd.DataFrame(players_json_list)
    df_metadata['id'] = pd.to_numeric(df_metadata['id'], errors='coerce').astype('Int64')
    
    # ⚠️ ATENÇÃO: Ajuste 'position' para o nome exato da chave de posição no seu JSON
    # Se for um dicionário aninhado (ex: role -> name), você pode precisar achatar antes.
    coluna_posicao = 'positionGroupType' # Pode ser 'role', 'position_name', etc.
    
    if coluna_posicao in df_metadata.columns:
        # Pega os IDs dos jogadores que SÃO goleiros (ajuste 'Goalkeeper' se estiver em outro idioma)
        goleiros_ids = df_metadata[df_metadata[coluna_posicao].astype(str).str.contains('Goalkeeper|Goleiro|GK', case=False, na=False)]['id'].tolist()
    else:
        print(f"⚠️ Coluna de posição '{coluna_posicao}' não encontrada no JSON. Omitindo filtro de goleiros.")
        goleiros_ids = []

    # ==========================================
    # 3. Aplicar os Filtros no DataFrame Base
    # ==========================================
    df_filtrado = pd.merge(df_rapm_base, df_minutos, on='player_id', how='inner')
    
    # Filtra Minutagem
    df_filtrado = df_filtrado[df_filtrado['minutos'] >= min_minutos].copy()
    
    # Filtra Goleiros
    df_filtrado = df_filtrado[~df_filtrado['player_id'].isin(goleiros_ids)].copy()
    
    # ==========================================
    # 4. Recalcular Z-Scores (Apenas para o grupo limpo)
    # ==========================================
    # Precisamos usar os valores puros do RAPM que geramos lá atrás
    cols_rapm = ['RAPM_xG', 'RAPM_xT', 'RAPM_VAEP']
    
    for col in cols_rapm:
        if col in df_filtrado.columns:
            df_filtrado[f'Z_{col}'] = stats.zscore(df_filtrado[col])
            
    # ==========================================
    # 5. Criar o Score Ponderado por Especialista (Substituindo o PCA)
    # ==========================================
    # Você define a importância de cada métrica para a substituição.
    # Exemplo: 20% para Finalização (xG), 40% para Criação (xT), 40% para Ação Geral (VAEP)
    peso_xg = 0.20
    peso_xt = 0.40
    peso_vaep = 0.40

    df_filtrado['Z_RAPM_xG'] = df_filtrado['Z_RAPM_xG'].clip(lower=-3, upper=3)
    df_filtrado['Z_RAPM_xT'] = df_filtrado['Z_RAPM_xT'].clip(lower=-3, upper=3)
    df_filtrado['Z_RAPM_VAEP'] = df_filtrado['Z_RAPM_VAEP'].clip(lower=-3, upper=3)
    
    # Todos os pesos somam e SÃO POSITIVOS. Nenhum jogador será punido por ser bom.
    df_filtrado['Raw_Score'] = (
        (df_filtrado['Z_RAPM_xG'] * peso_xg) +
        (df_filtrado['Z_RAPM_xT'] * peso_xt) +
        (df_filtrado['Z_RAPM_VAEP'] * peso_vaep)
    )

    # Escalar de 0 a 100 para leitura fácil
    scaler = MinMaxScaler(feature_range=(0, 100))
    df_filtrado['Sub_Impact_Score'] = scaler.fit_transform(df_filtrado[['Raw_Score']]).round(1)
    
    # Limpamos a coluna temporária
    df_filtrado = df_filtrado.drop(columns=['Raw_Score'])

    # ==========================================
    # 6. Resultados (Atualizado)
    # ==========================================
    print("\n⚖️ PESOS DO ÍNDICE (Definidos pelo Analista):")
    print(f"🔹 xG: {peso_xg*100:.0f}%")
    print(f"🔹 xT: {peso_xt*100:.0f}%")
    print(f"🔹 VAEP: {peso_vaep*100:.0f}%")

    df_final = df_filtrado.sort_values(by='Sub_Impact_Score', ascending=False).reset_index(drop=True)
    
    colunas_exibicao = ['player_name', 'minutos', 'Sub_Impact_Score', 'Z_RAPM_xG', 'Z_RAPM_xT', 'Z_RAPM_VAEP']
    
    print(f"\n🏆 TOP 10 JOGADORES (Amostra: {len(df_final)} jogadores avaliados):")
    display(df_final[colunas_exibicao].head(10))

    print("\n🔻 BOTTOM 5 JOGADORES:")
    display(df_final[colunas_exibicao].tail(5))
    
    return df_final

# ==========================================
# CHAMADA DA FUNÇÃO
# ==========================================
# Passe o df_resultados_rapm_limpo original (sem os z-scores antigos), o seu df_rapm_ready e a lista do JSON.
# Altere o 5 para 45 ou 90 depois para ver como o ranking se consolida!

df_ranking_final = gerar_ranking_pca(
    df_rapm_base=df_resultados_rapm_limpo, 
    df_janelas=df_rapm_ready, 
    players_json_list=players_list, 
    min_minutos=0
)

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import scipy.stats as stats

print("="*60)
print("🔄 AVALIAÇÃO DE EVENTOS: O SCORE DA SUBSTITUIÇÃO")
print("="*60)

# 1. Ordenar as janelas cronologicamente para cada time em cada jogo
# Garante que a janela 2 venha logo após a janela 1
df_eventos = df_pm_avancado.sort_values(by=['match_id', 'team_id', 'window_id']).copy()

# 2. Pegar o ritmo do time ANTES da substituição
# A função shift(1) "desce" a linha anterior, permitindo comparar o antes e o depois na mesma linha
df_eventos['pm_xg_p90_antes'] = df_eventos.groupby(['match_id', 'team_id'])['pm_xg_p90'].shift(1)
df_eventos['pm_xt_p90_antes'] = df_eventos.groupby(['match_id', 'team_id'])['pm_xt_p90'].shift(1)
df_eventos['pm_vaep_p90_antes'] = df_eventos.groupby(['match_id', 'team_id'])['pm_vaep_p90'].shift(1)

# A primeira janela do jogo (os titulares) não tem "antes", então dropamos para avaliar apenas substituições
df_eventos = df_eventos.dropna(subset=['pm_xg_p90_antes']).copy()

# Opcional: Filtrar janelas extremamente curtas (< 3 min) para evitar distorções
df_eventos = df_eventos[df_eventos['duracao_janela_min'] >= 10.0].copy()

# 3. Calcular o DELTA (A mudança real gerada pela alteração)
df_eventos['delta_xg'] = df_eventos['pm_xg_p90'] - df_eventos['pm_xg_p90_antes']
df_eventos['delta_xt'] = df_eventos['pm_xt_p90'] - df_eventos['pm_xt_p90_antes']
df_eventos['delta_vaep'] = df_eventos['pm_vaep_p90'] - df_eventos['pm_vaep_p90_antes']

# 4. Padronizar os Deltas (Z-Score)
# Aplicamos a clipagem (-3 a +3) diretamente aqui para blindar a régua de 0 a 100 contra anomalias (ex: lesões no minuto 89)
for col in ['delta_xg', 'delta_xt', 'delta_vaep']:
    z_col = f'z_{col}'
    df_eventos[z_col] = stats.zscore(df_eventos[col])
    df_eventos[z_col] = df_eventos[z_col].clip(lower=-3, upper=3)

# 5. Aplicar o Índice Ponderado pelo Especialista
# Mantemos a mesma lógica que validamos: xT e VAEP são mais estáveis e explicam melhor o controle do jogo
peso_xg = 0.20
peso_xt = 0.40
peso_vaep = 0.40

df_eventos['Raw_Event_Score'] = (
    (df_eventos['z_delta_xg'] * peso_xg) +
    (df_eventos['z_delta_xt'] * peso_xt) +
    (df_eventos['z_delta_vaep'] * peso_vaep)
)

# 6. Escalar para 0 a 100
scaler = MinMaxScaler(feature_range=(0, 100))
df_eventos['Event_Score'] = scaler.fit_transform(df_eventos[['Raw_Event_Score']]).round(1)

# 7. Organizar e Exibir
# Juntamos as colunas de "players_in" e "players_out" se elas existirem no seu df_pm_avancado
colunas_exibicao = [
    'match_id', 'team_id', 'minuto_sub', 'duracao_janela_min', 
    'delta_xg', 'delta_xt', 'delta_vaep', 'Event_Score'
]

# Garantir que exibe apenas colunas que realmente existem
colunas_exibicao = [col for col in colunas_exibicao if col in df_eventos.columns]

print("\n🏆 TOP 5 SUBSTITUIÇÕES DA TEMPORADA (Maior Mudança Positiva de Ritmo):")
display(df_eventos.sort_values(by='Event_Score', ascending=False)[colunas_exibicao].head())

print("\n🔻 BOTTOM 5 SUBSTITUIÇÕES DA TEMPORADA (Maior Piora no Ritmo):")
display(df_eventos.sort_values(by='Event_Score', ascending=True)[colunas_exibicao].head())

In [ ]:
import pandas as pd
import numpy as np

print("="*60)
print("➕ ENRIQUECENDO DADOS: POSIÇÕES E PLACARES")
print("="*60)

# ==========================================
# 1. Recuperar 'players_in' e 'players_out'
# ==========================================
# Fazemos um merge com o seu df_windows original para trazer de volta quem entrou/saiu e o tempo exato (t_start)
df_eventos_enriquecido = pd.merge(
    df_eventos,
    df_windows[['match_id', 'team_id', 'window_id', 'players_in', 'players_out', 't_start']],
    on=['match_id', 'team_id', 'window_id'],
    how='left'
)

# ==========================================
# 2. Mapear IDs para Posições
# ==========================================
df_meta = pd.DataFrame(players_list)

# Garante que os IDs estão no mesmo formato (Int64)
ids_json = pd.to_numeric(df_meta['id'], errors='coerce').astype('Int64')

# ⚠️ ATENÇÃO: Verifique se a chave da posição no seu JSON é 'position', 'role' ou outra.
chave_posicao = 'positionGroupType' 
dict_posicoes = dict(zip(ids_json, df_meta[chave_posicao]))

def mapear_posicoes(lista_ids):
    if not isinstance(lista_ids, list): 
        return "N/A"
    
    posicoes = []
    for p in lista_ids:
        p_num = pd.to_numeric(p, errors='coerce')
        # Pega a posição no dicionário. Se não achar, retorna 'Desconhecido'
        pos = str(dict_posicoes.get(p_num, 'Desconhecido'))
        posicoes.append(pos)
        
    return " + ".join(posicoes)

# Aplica a função para criar as novas colunas
df_eventos_enriquecido['posicao_in'] = df_eventos_enriquecido['players_in'].apply(mapear_posicoes)
df_eventos_enriquecido['posicao_out'] = df_eventos_enriquecido['players_out'].apply(mapear_posicoes)

# ==========================================
# 3. Calcular os Placares (Momento da Sub e Final)
# ==========================================
# Isolamos apenas os eventos de gol para otimizar o processamento
df_gols = actions_df[pd.to_numeric(actions_df['goal'], errors='coerce') == 1].copy()

def calcular_placares(row):
    match = row['match_id']
    team = row['team_id']
    t_sub = row['t_start']
    
    # Filtra os gols desta partida específica
    gols_partida = df_gols[df_gols['match_id'] == match]
    
    # Separa os gols de quem fez a substituição (Time) e quem sofreu (Adversário)
    gols_time = gols_partida[gols_partida['team_id'] == team]
    gols_adv = gols_partida[gols_partida['team_id'] != team]
    
    # Placar no MOMENTO da substituição (Gols que aconteceram antes do t_start)
    score_time_momento = len(gols_time[gols_time['start_time'] < t_sub])
    score_adv_momento = len(gols_adv[gols_adv['start_time'] < t_sub])
    
    # Placar FINAL da partida
    score_time_final = len(gols_time)
    score_adv_final = len(gols_adv)
    
    return pd.Series({
        'placar_momento': f"{score_time_momento} x {score_adv_momento}",
        'placar_final': f"{score_time_final} x {score_adv_final}"
    })

# Aplica o cálculo linha a linha
placares = df_eventos_enriquecido.apply(calcular_placares, axis=1)
df_eventos_enriquecido = pd.concat([df_eventos_enriquecido, placares], axis=1)

# ==========================================
# 3.5. Determinar Mandante vs Visitante
# ==========================================
# Pegamos a primeira ação de cada time no jogo para checar o status de 'home_team'
df_mando = actions_df.drop_duplicates(subset=['match_id', 'team_id'])[['match_id', 'team_id', 'home_team']]

# Criamos um dicionário para busca rápida: chave = (match_id, team_id), valor = home_team
mando_dict = df_mando.set_index(['match_id', 'team_id'])['home_team'].to_dict()

def identificar_mando(row):
    is_home = mando_dict.get((row['match_id'], row['team_id']))
    
    # Checagem robusta (caso o JSON/provedor use 'True', 1, booleano nativo, etc.)
    if str(is_home).strip().lower() in ['true', '1', 'yes', 't']:
        return 'Mandante'
    elif str(is_home).strip().lower() in ['false', '0', 'no', 'f']:
        return 'Visitante'
    else:
        return 'Desconhecido'

df_eventos_enriquecido['mando_campo'] = df_eventos_enriquecido.apply(identificar_mando, axis=1)

# ==========================================
# 4. Exibir Resultado Final (ATUALIZADO COMPLETO)
# ==========================================
# Inserimos 'mando_campo' logo no começo para contextualizar a substituição
colunas_finais = [
    'match_id', 'mando_campo', 'minuto_sub', 
    'placar_momento', 'posicao_out', 'posicao_in', 
    'delta_xg', 'delta_xt', 'delta_vaep', 'Event_Score', 'placar_final'
]

# Garantir que exibe apenas colunas que realmente existem
colunas_finais = [col for col in colunas_finais if col in df_eventos_enriquecido.columns]

print("✅ Dataframe enriquecido com Mando de Campo!\n")

print("🏆 TOP 5 SUBSTITUIÇÕES (Maior Mudança Positiva):")
display(df_eventos_enriquecido.sort_values(by='Event_Score', ascending=False)[colunas_finais].head())

print("\n🔻 BOTTOM 5 SUBSTITUIÇÕES (Maior Piora no Ritmo):")
display(df_eventos_enriquecido.sort_values(by='Event_Score', ascending=True)[colunas_finais].head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import scipy.stats as stats

print("="*60)
print("🧠 AVALIAÇÃO DE EVENTOS INTELIGENTE (BAYESIANA + SCORE EFFECTS)")
print("="*60)

# ==========================================
# 1. Preparação e Ordenação
# ==========================================
df_eventos = df_pm_avancado.sort_values(by=['match_id', 'team_id', 'window_id']).copy()

# ==========================================
# 2. Suavização Bayesiana (Ritmo Acumulado do Jogo)
# ==========================================
# Em vez do turno anterior, somamos tudo que aconteceu no jogo até o momento da substituição
df_eventos['cum_mins'] = df_eventos.groupby(['match_id', 'team_id'])['duracao_janela_min'].cumsum() - df_eventos['duracao_janela_min']

df_eventos['cum_xg_raw'] = df_eventos.groupby(['match_id', 'team_id'])['pm_xg_raw'].cumsum() - df_eventos['pm_xg_raw']
df_eventos['cum_xt_raw'] = df_eventos.groupby(['match_id', 'team_id'])['pm_xt_raw'].cumsum() - df_eventos['pm_xt_raw']
df_eventos['cum_vaep_raw'] = df_eventos.groupby(['match_id', 'team_id'])['pm_vaep_raw'].cumsum() - df_eventos['pm_vaep_raw']

# Calculamos o P90 Histórico ("O Ritmo do Jogo Antes da Sub")
df_eventos['xg_antes'] = np.where(df_eventos['cum_mins'] > 0, (df_eventos['cum_xg_raw'] / df_eventos['cum_mins']) * 90, np.nan)
df_eventos['xt_antes'] = np.where(df_eventos['cum_mins'] > 0, (df_eventos['cum_xt_raw'] / df_eventos['cum_mins']) * 90, np.nan)
df_eventos['vaep_antes'] = np.where(df_eventos['cum_mins'] > 0, (df_eventos['cum_vaep_raw'] / df_eventos['cum_mins']) * 90, np.nan)

# Filtro Anti-Ruído: Remover primeira janela e janelas onde o jogador avaliado jogou menos de 10 min
df_eventos = df_eventos.dropna(subset=['xg_antes']).copy()
df_eventos = df_eventos[df_eventos['duracao_janela_min'] >= 10.0].copy()

# Calculamos o Delta Bruto
df_eventos['delta_xg_bruto'] = df_eventos['pm_xg_p90'] - df_eventos['xg_antes']
df_eventos['delta_xt_bruto'] = df_eventos['pm_xt_p90'] - df_eventos['xt_antes']
df_eventos['delta_vaep_bruto'] = df_eventos['pm_vaep_p90'] - df_eventos['vaep_antes']

# ==========================================
# 3. Contexto: Placares, Posições e Mando
# ==========================================
# Merge de contexto do df_windows
df_eventos = pd.merge(df_eventos, df_windows[['match_id', 'team_id', 'window_id', 'players_in', 'players_out', 't_start']], on=['match_id', 'team_id', 'window_id'], how='left')

# Mando de Campo
df_mando = actions_df.drop_duplicates(subset=['match_id', 'team_id'])[['match_id', 'team_id', 'home_team']]
mando_dict = df_mando.set_index(['match_id', 'team_id'])['home_team'].to_dict()
df_eventos['mando_campo'] = df_eventos.apply(lambda r: 'Mandante' if str(mando_dict.get((r['match_id'], r['team_id']))).strip().lower() in ['true', '1', 't'] else 'Visitante', axis=1)

# Placares e Saldo de Gols
df_gols = actions_df[pd.to_numeric(actions_df['goal'], errors='coerce') == 1].copy()

def calcular_contexto_placar(row):
    gols_partida = df_gols[df_gols['match_id'] == row['match_id']]
    gols_time = gols_partida[gols_partida['team_id'] == row['team_id']]
    gols_adv = gols_partida[gols_partida['team_id'] != row['team_id']]
    
    score_time_mom = len(gols_time[gols_time['start_time'] < row['t_start']])
    score_adv_mom = len(gols_adv[gols_adv['start_time'] < row['t_start']])
    
    return pd.Series({
        'placar_momento': f"{score_time_mom} x {score_adv_mom}",
        'saldo_gols_momento': score_time_mom - score_adv_mom, # GD (Goal Difference)
        'placar_final': f"{len(gols_time)} x {len(gols_adv)}"
    })

df_eventos = pd.concat([df_eventos, df_eventos.apply(calcular_contexto_placar, axis=1)], axis=1)

# Posições (Garante que dict_posicoes existe do código anterior)
df_meta = pd.DataFrame(players_list)
ids_json = pd.to_numeric(df_meta['id'], errors='coerce').astype('Int64')
chave_posicao = 'positionGroupType' if 'positionGroupType' in df_meta.columns else 'nickname' # Fallback
dict_posicoes = dict(zip(ids_json, df_meta[chave_posicao]))

def map_pos(lista_ids):
    if not isinstance(lista_ids, list): return "N/A"
    return " + ".join([str(dict_posicoes.get(pd.to_numeric(p, errors='coerce'), '?')) for p in lista_ids])

df_eventos['posicao_out'] = df_eventos['players_out'].apply(map_pos)
df_eventos['posicao_in'] = df_eventos['players_in'].apply(map_pos)

# ==========================================
# 4. Ajuste pelo Estado do Jogo (Score Effects)
# ==========================================
def aplicar_score_effects(row):
    gd = row['saldo_gols_momento']
    
    def ajustar_delta(delta):
        if gd < 0: # Time perdendo (Atacar é mais fácil porque o adversário recua)
            # Desconto o mérito se for positivo. Puno mais se for negativo.
            return delta * 0.85 if delta > 0 else delta * 1.15
        elif gd > 0: # Time ganhando (Atacar é difícil porque o adversário pressiona)
            # Aumento o mérito se for positivo. Perdôo um pouco se for negativo.
            return delta * 1.15 if delta > 0 else delta * 0.85
        return delta # Empate (Sem ajuste)
        
    return pd.Series({
        'delta_xg': ajustar_delta(row['delta_xg_bruto']),
        'delta_xt': ajustar_delta(row['delta_xt_bruto']),
        'delta_vaep': ajustar_delta(row['delta_vaep_bruto'])
    })

df_eventos = pd.concat([df_eventos, df_eventos.apply(aplicar_score_effects, axis=1)], axis=1)

# ==========================================
# 5. Z-Score, Clipping e Score Final
# ==========================================
# Aplicamos a clipagem rigorosa de anomalias (-3 a +3 desvios padrões)
for col in ['delta_xg', 'delta_xt', 'delta_vaep']:
    z_col = f'z_{col}'
    df_eventos[z_col] = stats.zscore(df_eventos[col])
    df_eventos[z_col] = df_eventos[z_col].clip(lower=-3, upper=3)

# O Índice do Especialista (Pesos justos)
df_eventos['Raw_Score'] = (df_eventos['z_delta_xg']*0.20) + (df_eventos['z_delta_xt']*0.40) + (df_eventos['z_delta_vaep']*0.40)

scaler = MinMaxScaler(feature_range=(0, 100))
df_eventos['Event_Score'] = scaler.fit_transform(df_eventos[['Raw_Score']]).round(1)

# ==========================================
# 6. Exibição
# ==========================================
colunas_exibicao = [
    'match_id', 'mando_campo', 'minuto_sub', 'duracao_janela_min',
    'placar_momento', 'posicao_out', 'posicao_in', 
    'delta_xg', 'delta_xt', 'delta_vaep', 'Event_Score', 'placar_final'
]

print("✅ Modelo reconstruído e livre de ruídos!\n")

print("🏆 TOP 5 SUBSTITUIÇÕES TÁTICAS REAIS (Impacto Ajustado):")
display(df_eventos.sort_values(by='Event_Score', ascending=False)[colunas_exibicao].head())

print("\n🔻 BOTTOM 5 SUBSTITUIÇÕES TÁTICAS REAIS (Piora Ajustada):")
display(df_eventos.sort_values(by='Event_Score', ascending=True)[colunas_exibicao].head())